In [1]:
import numpy as np
from scipy.spatial.distance import cdist
from scipy.optimize import minimize
import copy

class Node:
    def __init__(self, id, pos, left=None, right=None):
        self.id = id
        self.pos = np.array(pos, dtype=float)
        self.left = left
        self.right = right
        self.parent = None  # To track the connection upwards

    def is_leaf(self):
        return self.left is None and self.right is None

def total_tree_length(node):
    """Recursively calculates the total edge length of the tree."""
    if node is None:
        return 0
    length = 0
    if node.left:
        length += np.linalg.norm(node.pos - node.left.pos) + total_tree_length(node.left)
    if node.right:
        length += np.linalg.norm(node.pos - node.right.pos) + total_tree_length(node.right)
    return length

def build_topology(leaves):
    """
    Phase 1: Greedy Agglomerative Clustering (Neighbor Joining).
    Connects closest nodes until 1 root remains.
    """
    # Create a working list of active nodes
    active_nodes = [Node(i, coord) for i, coord in enumerate(leaves)]
    next_id = len(leaves)

    while len(active_nodes) > 1:
        # Extract positions of all active nodes
        positions = np.array([n.pos for n in active_nodes])
        
        # Calculate distance matrix (add infinity to diagonal to ignore self-distance)
        dists = cdist(positions, positions)
        np.fill_diagonal(dists, np.inf)
        
        # Find indices of the closest pair
        min_idx = np.unravel_index(np.argmin(dists), dists.shape)
        idx_a, idx_b = min_idx
        
        node_a = active_nodes[idx_a]
        node_b = active_nodes[idx_b]
        
        # Create new internal parent at the midpoint
        new_pos = (node_a.pos + node_b.pos) / 2.0
        parent = Node(next_id, new_pos, left=node_a, right=node_b)
        next_id += 1
        
        # Link children to parent
        node_a.parent = parent
        node_b.parent = parent
        
        # Remove old nodes, add new parent
        # (Remove larger index first to avoid shifting problems)
        active_nodes.pop(max(idx_a, idx_b))
        active_nodes.pop(min(idx_a, idx_b))
        active_nodes.append(parent)

    return active_nodes[0] # The Root

def optimize_geometry(root, iterations=10):
    """
    Phase 2: Iterative Relaxation.
    Moves internal nodes to the geometric median (Fermat Point) of their neighbors.
    """
    # Collect all internal nodes (excluding leaves)
    internal_nodes = []
    stack = [root]
    while stack:
        curr = stack.pop()
        if not curr.is_leaf():
            internal_nodes.append(curr)
            stack.append(curr.left)
            stack.append(curr.right)
    
    # Objective function for the Fermat Point:
    # Min(dist(P, Neighbor1) + dist(P, Neighbor2) + dist(P, Neighbor3))
    def local_energy(point, neighbors):
        return sum(np.linalg.norm(point - n) for n in neighbors)

    print(f"Initial Length: {total_tree_length(root):.4f}")

    for i in range(iterations):
        max_shift = 0
        
        for node in internal_nodes:
            # Gather neighbor positions
            neighbors = []
            if node.left: neighbors.append(node.left.pos)
            if node.right: neighbors.append(node.right.pos)
            if node.parent: neighbors.append(node.parent.pos)
            
            # If root (no parent), it only has 2 neighbors. 
            # Optimal position is strictly on the line segment between them.
            # If internal node, it has 3 neighbors.
            
            if len(neighbors) < 2: continue 

            # Find optimal position for this single node relative to its neighbors
            res = minimize(local_energy, node.pos, args=(neighbors,), method='L-BFGS-B')
            
            shift = np.linalg.norm(node.pos - res.x)
            node.pos = res.x
            if shift > max_shift:
                max_shift = shift
        
        print(f"Iteration {i+1}: Max Shift = {max_shift:.6f}, Total Length = {total_tree_length(root):.4f}")
        
        if max_shift < 1e-4: # Convergence check
            break
            
    return root


def build_topology_balanced(leaves):
    """
    Phase 1 (Balanced): Layer-by-layer construction.
    Pairs all nodes in the current layer before moving up.
    """
    # Initialize Layer 0 with all leaves
    current_layer = [Node(i, coord) for i, coord in enumerate(leaves)]
    next_id = len(leaves)

    while len(current_layer) > 1:
        next_layer = []
        
        # 1. Calculate distance matrix for the CURRENT layer only
        positions = np.array([n.pos for n in current_layer])
        dists = cdist(positions, positions)
        np.fill_diagonal(dists, np.inf)
        
        # Keep track of who has been paired in this round
        paired_indices = set()
        
        # 2. Iteratively find closest pairs until we run out
        # (We loop enough times to cover worst-case pairings)
        for _ in range(len(current_layer) // 2):
            # Mask out rows/cols of already paired nodes
            # (We set them to infinity so they aren't picked again)
            masked_dists = dists.copy()
            if paired_indices:
                idx_list = list(paired_indices)
                masked_dists[idx_list, :] = np.inf
                masked_dists[:, idx_list] = np.inf
            
            # Find the global minimum in the masked matrix
            min_val = np.min(masked_dists)
            if min_val == np.inf:
                break # No more valid pairs
                
            min_idx = np.unravel_index(np.argmin(masked_dists), masked_dists.shape)
            idx_a, idx_b = min_idx
            
            # Create the parent
            node_a = current_layer[idx_a]
            node_b = current_layer[idx_b]
            
            new_pos = (node_a.pos + node_b.pos) / 2.0
            parent = Node(next_id, new_pos, left=node_a, right=node_b)
            node_a.parent = parent
            node_b.parent = parent
            
            # Add parent to next layer
            next_layer.append(parent)
            next_id += 1
            
            # Mark these indices as processed
            paired_indices.add(idx_a)
            paired_indices.add(idx_b)

        # 3. Handle the "Straggler" (if odd number of nodes)
        # Any node not in paired_indices gets promoted to next layer alone
        for i in range(len(current_layer)):
            if i not in paired_indices:
                next_layer.append(current_layer[i])
        
        # Move up to the next level
        current_layer = next_layer

    return current_layer[0] # The Root


In [2]:
np.random.seed(42)
leaves_coords = np.random.rand(16, 3) * 100 # Use power of 2 for perfect balance

print("Building Balanced Topology...")
tree_root = build_topology_balanced(leaves_coords)

print("Optimizing Geometry...")
tree_root = optimize_geometry(tree_root)

Building Balanced Topology...
Optimizing Geometry...
Initial Length: 621.6190
Iteration 1: Max Shift = 26.522620, Total Length = 529.9151
Iteration 2: Max Shift = 4.736241, Total Length = 527.4810
Iteration 3: Max Shift = 1.705174, Total Length = 526.9038
Iteration 4: Max Shift = 0.671661, Total Length = 526.5873
Iteration 5: Max Shift = 0.570046, Total Length = 526.3196
Iteration 6: Max Shift = 0.600549, Total Length = 526.0676
Iteration 7: Max Shift = 0.609526, Total Length = 525.8322
Iteration 8: Max Shift = 0.603796, Total Length = 525.6230
Iteration 9: Max Shift = 0.554275, Total Length = 525.4853
Iteration 10: Max Shift = 0.199893, Total Length = 525.4606
